# CineMate – Agentic Movie Recommendation Assistant
## Phase 2 Live Integration

Live end-to-end run of the CineMate code from GitHub:

```
User -> PromptTemplate -> LangChain Agent (Gemini) -> search_movies Tool -> TMDB API -> movie data -> Gemini -> recommendation
```

**Before running (Kaggle):**
1. Notebook settings -> **Internet: On** (needed to clone the repo and reach TMDB/Gemini).
2. Add-ons -> **Secrets**: create `TMDB_API_KEY`, `GEMINI_API_KEY`, `GEMINI_MODEL` and toggle each one **attached** to this notebook.
3. Run all cells. No code edits needed. **Never paste keys into cells.**

Gemini free tier has a small daily request quota per model. Each Gemini section stops on `429`/`503` instead of retrying.

## 1. Project Introduction

CineMate is a LangChain agent. The LLM (Gemini) decides when it needs movie facts, calls the `search_movies` tool,
which queries the real TMDB API, and then writes a recommendation grounded in the retrieved data.
This notebook runs the actual repository code (`src/cinemate`), not a copy. Kaggle is used only as the live
execution environment for Phase 2, because the development laptop's network cannot resolve TMDB correctly.

## 2. Environment Setup

In [ ]:
import os, sys, subprocess, platform

REPO_URL = "https://github.com/Malik8122/cinemate-movie-agent.git"
REPO_DIR = "/kaggle/working/cinemate-movie-agent" if os.path.exists("/kaggle") else os.path.abspath("cinemate-movie-agent")

if os.path.isdir(os.path.join(REPO_DIR, ".git")):
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], check=False)
else:
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)

if not os.path.exists(os.path.join(REPO_DIR, "requirements.txt")):
    raise RuntimeError("Repository cloned but requirements.txt is missing: the GitHub repo is empty or the code has not been pushed yet.")

sys.path.insert(0, os.path.join(REPO_DIR, "src"))   # make `import cinemate` use the repository code
print("Python  :", platform.python_version())
print("Repo    :", REPO_DIR)
print("Commit  :", subprocess.run(["git", "-C", REPO_DIR, "log", "-1", "--oneline"], capture_output=True, text=True).stdout.strip())

results = {}   # test name -> PASS / FAIL / SKIPPED, printed in the final section

## 3. Secret Configuration
Secrets are read from Kaggle Secrets (or existing environment variables). Values are never printed.

In [ ]:
secret_errors = {}

def load_secret(name):
    if os.getenv(name):
        return
    try:
        from kaggle_secrets import UserSecretsClient
        os.environ[name] = UserSecretsClient().get_secret(name)
    except Exception as e:   # not on Kaggle, or secret missing / not attached to this notebook
        secret_errors[name] = f"{type(e).__name__}: {str(e)[:150]}"

for name in ("TMDB_API_KEY", "GEMINI_API_KEY", "GEMINI_MODEL"):
    load_secret(name)

SECRET_VALUES = [os.environ[n] for n in ("TMDB_API_KEY", "GEMINI_API_KEY") if os.getenv(n)]

def scrub(text):
    """Remove any secret value from text before displaying it."""
    text = str(text)
    for s in SECRET_VALUES:
        text = text.replace(s, "<hidden>")
    return text

print("TMDB_API_KEY configured  :", bool(os.getenv("TMDB_API_KEY")))
print("GEMINI_API_KEY configured:", bool(os.getenv("GEMINI_API_KEY")))
print("GEMINI_MODEL             :", os.getenv("GEMINI_MODEL") or "NOT CONFIGURED")
for name, err in secret_errors.items():
    print(f"  {name} not loaded -> {err}")

## 4. Dependency Installation
Checks installed versions against the repository's pinned `requirements.txt` and installs only if something differs.

In [ ]:
from importlib.metadata import version, PackageNotFoundError

def pinned():
    for line in open(os.path.join(REPO_DIR, "requirements.txt")):
        line = line.strip()
        if line and not line.startswith("#") and "==" in line:
            yield line.split("==")

def installed(pkg):
    try:
        return version(pkg)
    except PackageNotFoundError:
        return None

mismatch = [(p, v, installed(p)) for p, v in pinned() if installed(p) != v]
if mismatch:
    print("Installing pinned requirements (differences:", [(p, have) for p, _, have in mismatch], ")")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", os.path.join(REPO_DIR, "requirements.txt")], check=False)
    print("If versions below still look wrong: Session -> Restart, then Run All.")
else:
    print("Installed packages already match requirements.txt")

print()
print("Python           :", platform.python_version())
for p in ("langchain", "langchain-core", "langchain-google-genai", "pydantic"):
    print(f"{p:<25}:", installed(p))

## 5. Configuration Verification

In [ ]:
from cinemate.config import ConfigError, get_tmdb_api_key, get_gemini_api_key, get_model_name

config_ok = True
for getter in (get_tmdb_api_key, get_gemini_api_key, get_model_name):
    try:
        getter()
        print(f"{getter.__name__}: OK")
    except ConfigError as e:
        config_ok = False
        print(e)
tmdb_configured = bool(os.getenv("TMDB_API_KEY"))
gemini_configured = bool(os.getenv("GEMINI_API_KEY") and os.getenv("GEMINI_MODEL"))
results["Configuration"] = "PASS" if config_ok else "FAIL"

## Test 1 – TMDB API Integration
Direct call to the repository's TMDB client: `Kaggle -> Python -> TMDB client -> TMDB API -> real response`.

In [ ]:
import socket
from cinemate.tmdb_client import search_movies as tmdb_search

print("api.themoviedb.org resolves to:", sorted({r[4][0] for r in socket.getaddrinfo("api.themoviedb.org", 443, proto=socket.IPPROTO_TCP)}))

tmdb_ok = False
if not tmdb_configured:
    print("SKIPPED: TMDB_API_KEY is not configured.")
    results["Test 1 - TMDB search ('Inception')"] = "SKIPPED"
else:
    r = tmdb_search("Inception")
    print("status :", r.status)
    print("message:", r.message)
    for m in r.movies[:3]:
        print(f"\n{m.title} ({m.year})  rating {m.rating}/10  genres: {', '.join(m.genres)}")
        print("  ", m.overview[:200])
    tmdb_ok = r.success and any("inception" in m.title.lower() for m in r.movies)
    results["Test 1 - TMDB search ('Inception')"] = "PASS" if tmdb_ok else "FAIL"

### Test 1b – No-result handling
A query that matches nothing must return a safe message, not an invented movie.

In [ ]:
if not tmdb_configured:
    print("SKIPPED: TMDB_API_KEY is not configured.")
    results["Test 1b - No results ('XYZABC123456789')"] = "SKIPPED"
else:
    r = tmdb_search("XYZABC123456789")
    print("status :", r.status)
    print("message:", r.message)
    print("movies :", r.movies)
    results["Test 1b - No results ('XYZABC123456789')"] = "PASS" if (r.status == "no_results" and r.movies == []) else "FAIL"

## Test 2 – Gemini Integration
One minimal request, **no retries**. On `429`/`503` the section reports the error and stops.

In [ ]:
from cinemate.agent import get_llm

gemini_ok = False
if not gemini_configured:
    print("SKIPPED: GEMINI_API_KEY / GEMINI_MODEL not configured.")
    results["Test 2 - Gemini response"] = "SKIPPED"
else:
    try:
        reply = get_llm(retries=0).invoke("Respond with exactly: CineMate Gemini test successful.")
        print("Model  :", os.getenv("GEMINI_MODEL"))
        print("Reply  :", reply.text)
        gemini_ok = "CineMate Gemini test successful" in reply.text
    except Exception as e:
        print(f"Gemini call failed: {type(e).__name__}")
        print(scrub(e)[:700])
        print("\n429 RESOURCE_EXHAUSTED = daily free-tier quota for this model is used up -> set GEMINI_MODEL to another model.")
        print("503 UNAVAILABLE = temporary capacity problem -> wait and re-run this cell once. Not retrying automatically.")
    results["Test 2 - Gemini response"] = "PASS" if gemini_ok else "FAIL"

## Test 3 – LangChain Tool
The `search_movies` tool, and whether Gemini can request it through LangChain tool binding: `Gemini -> tool binding -> search_movies`.

In [ ]:
from cinemate.tools import search_movies

print("Tool name       :", search_movies.name)
print("Tool description:", " ".join(search_movies.description.split()))
print("Tool arguments  :", list(search_movies.args))
print("\n--- Tool invoked directly: search_movies('Inception') ---")
if not tmdb_configured:
    print("SKIPPED: TMDB_API_KEY is not configured.")
    results["Test 3a - LangChain tool -> TMDB"] = "SKIPPED"
else:
    tool_output = search_movies.invoke({"query": "Inception"})
    print(tool_output[:700])
    results["Test 3a - LangChain tool -> TMDB"] = "PASS" if "Inception" in tool_output else "FAIL"

In [ ]:
tool_calling_ok = False
if not gemini_ok:
    print("SKIPPED: Gemini did not respond in Test 2.")
    results["Test 3b - Gemini tool calling"] = "SKIPPED"
else:
    try:
        reply = get_llm(retries=0).bind_tools([search_movies]).invoke(
            "Use the movie search tool to look up the movie Inception.")
        print("Tool calls requested by Gemini:", reply.tool_calls)
        tool_calling_ok = bool(reply.tool_calls) and reply.tool_calls[0]["name"] == "search_movies"
    except Exception as e:
        print(f"Gemini call failed: {type(e).__name__}")
        print(scrub(e)[:700])
    if not tool_calling_ok:
        print("\nIf the model answered without a tool call, try another Gemini model in the GEMINI_MODEL secret.")
    results["Test 3b - Gemini tool calling"] = "PASS" if tool_calling_ok else "FAIL"

## 9. Agent
Build the agent with LangChain 1.x `create_agent`: Gemini + `search_movies` + the CineMate system prompt.

In [ ]:
from cinemate.agent import build_agent, run, print_trace, final_answer, tools_called
from cinemate.prompts import SYSTEM_PROMPT

agent_ready = tmdb_ok and gemini_ok and tool_calling_ok
if agent_ready:
    agent = build_agent()
    print("Agent built:", type(agent).__name__)
else:
    print("SKIPPED: TMDB, Gemini and tool calling must all pass first.")
    print({"tmdb": tmdb_ok, "gemini": gemini_ok, "tool_calling": tool_calling_ok})
print("\n--- CineMate system prompt ---\n" + SYSTEM_PROMPT)

## Test 4 – Agentic Movie Recommendation
```
User -> Gemini Agent -> search_movies -> TMDB API -> movie information -> Gemini -> recommendation
```
Each step is printed. Text under **[TMDB]** is retrieved data; text under **[CINEMATE]** is the LLM's recommendation.

In [ ]:
def run_agent_test(name, question, check):
    if not agent_ready:
        print("SKIPPED: prerequisites not met.")
        results[name] = "SKIPPED"
        return
    try:
        messages = run(agent, question)
        print_trace(messages)
        results[name] = "PASS" if check(messages) else "FAIL"
    except Exception as e:
        print(f"Agent run failed: {type(e).__name__}")
        print(scrub(e)[:700])
        print("\nOn 429/503: not retrying automatically (protects free-tier quota).")
        results[name] = "FAIL"

used_tool = lambda msgs: "search_movies" in tools_called(msgs) and len(final_answer(msgs)) > 0
run_agent_test("Test 4a - Agent: sci-fi recommendations", "Recommend some science-fiction movies.", used_tool)

### Test 4b – Natural-language preferences
The tool cannot filter by rating or date range; the agent must judge the retrieved data itself and say so.

In [ ]:
run_agent_test("Test 4b - Agent: thriller after 2020, rating > 7",
               "I want a thriller movie released after 2020 with a rating above 7.", used_tool)

## Test 5 – Error Handling
A movie that does not exist: CineMate must say it could not find it and must **not** invent one.

In [ ]:
def no_result_check(msgs):
    tool_texts = [m.text for m in msgs if m.type == "tool"]
    zero_results = bool(tool_texts) and all("No movies found" in t for t in tool_texts)
    print("Tool returned zero results:", zero_results, "-> read the [CINEMATE] answer above to confirm no movie was invented.")
    return zero_results

run_agent_test("Test 5 - Agent: nonexistent movie", "Recommend me a movie called XYZABC123456789.", no_result_check)

## 12. Results

In [ ]:
width = max(len(k) for k in results)
for name, outcome in results.items():
    print(f"{name:<{width}}  {outcome}")
print()
print("Passed :", sum(v == "PASS" for v in results.values()))
print("Failed :", sum(v == "FAIL" for v in results.values()))
print("Skipped:", sum(v == "SKIPPED" for v in results.values()))
print("GEMINI_MODEL used:", os.getenv("GEMINI_MODEL"))